# SmartPump AI — 01: Data Exploration & Statistical Analysis

**Step 2 of the SmartPump AI pipeline.**

This notebook performs a complete, reproducible Exploratory Data Analysis (EDA) of the **UCI Condition Monitoring of Hydraulic Systems** dataset.

**Objective:** Classify hydraulic pump internal leakage into three operational states:
- `0` → **Normal** (no leakage)
- `1` → **Warning** (weak leakage)
- `2` → **Fault** (severe leakage)

**Run after:** Downloading all 18 dataset files to `data/raw/`  
**Run before:** `02_preprocessing.ipynb`

---

## Notebook Contents

| Task | Description |
|------|-------------|
| **A** | Dataset Verification — files, shapes, label alignment |
| **B** | Statistical Analysis — descriptive stats per sensor and per class |
| **C** | Data Visualization — 9 publication-quality charts |
| **D** | Sensor Analysis — which channels discriminate pump leakage |
| **E** | Data Quality & Engineering Risks — missing values, outliers, leakage risks |
| **F** | Dataset Preparation Findings — structure for Step 3 preprocessing |

## Setup and Imports

In [ ]:
import os
import sys
import warnings
from pathlib import Path

os.environ["MPLCONFIGDIR"] = os.environ.get("TEMP", os.environ.get("TMP", "/tmp"))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as scipy_stats
from IPython.display import display

warnings.filterwarnings("ignore")

curr_p = Path(".").resolve()
if (curr_p / "data" / "raw").exists():
    PROJECT_ROOT = curr_p
elif (curr_p.parent / "data" / "raw").exists():
    PROJECT_ROOT = curr_p.parent
else:
    PROJECT_ROOT = curr_p

RAW_DIR       = PROJECT_ROOT / "data" / "raw"
EDA_DIR       = PROJECT_ROOT / "results" / "eda"
VIZ_DIR       = EDA_DIR / "visualizations"

EDA_DIR.mkdir(parents=True, exist_ok=True)
VIZ_DIR.mkdir(parents=True, exist_ok=True)

# ── Add src/ to path ─────────────────────────────────────────────────────────
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from preprocessing import (
    load_profile, load_sensor, load_all_sensors,
    EXPECTED_SHAPES, PUMP_LEAKAGE_LABEL_MAP, describe_dataset,
)

# ── Plot style ───────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 120,
    "savefig.bbox": "tight",
    "font.size": 10,
})

LABEL_MAP    = {0: "Normal", 1: "Warning", 2: "Fault"}
CLASS_COLORS = {"Normal": "#2196F3", "Warning": "#FF9800", "Fault": "#F44336"}
CLASS_ORDER  = ["Normal", "Warning", "Fault"]

# ── Sensor metadata ──────────────────────────────────────────────────────────
SENSOR_META = {
    "PS1":  {"unit": "bar",   "rate_hz": 100, "n_cols": 6000, "type": "Pressure"},
    "PS2":  {"unit": "bar",   "rate_hz": 100, "n_cols": 6000, "type": "Pressure"},
    "PS3":  {"unit": "bar",   "rate_hz": 100, "n_cols": 6000, "type": "Pressure"},
    "PS4":  {"unit": "bar",   "rate_hz": 100, "n_cols": 6000, "type": "Pressure"},
    "PS5":  {"unit": "bar",   "rate_hz": 100, "n_cols": 6000, "type": "Pressure"},
    "PS6":  {"unit": "bar",   "rate_hz": 100, "n_cols": 6000, "type": "Pressure"},
    "EPS1": {"unit": "W",     "rate_hz": 100, "n_cols": 6000, "type": "Motor Power"},
    "FS1":  {"unit": "l/min", "rate_hz":  10, "n_cols":  600, "type": "Flow"},
    "FS2":  {"unit": "l/min", "rate_hz":  10, "n_cols":  600, "type": "Flow"},
    "TS1":  {"unit": "°C",    "rate_hz":   1, "n_cols":   60, "type": "Temperature"},
    "TS2":  {"unit": "°C",    "rate_hz":   1, "n_cols":   60, "type": "Temperature"},
    "TS3":  {"unit": "°C",    "rate_hz":   1, "n_cols":   60, "type": "Temperature"},
    "TS4":  {"unit": "°C",    "rate_hz":   1, "n_cols":   60, "type": "Temperature"},
    "VS1":  {"unit": "mm/s",  "rate_hz":   1, "n_cols":   60, "type": "Vibration"},
    "CE":   {"unit": "%",     "rate_hz":   1, "n_cols":   60, "type": "Virtual"},
    "CP":   {"unit": "kW",    "rate_hz":   1, "n_cols":   60, "type": "Virtual"},
    "SE":   {"unit": "%",     "rate_hz":   1, "n_cols":   60, "type": "Virtual"},
}
SENSOR_ORDER = list(SENSOR_META.keys())

print(f"Project root : {PROJECT_ROOT}")
print(f"Raw data dir : {RAW_DIR}  (exists={RAW_DIR.exists()})")
print(f"EDA output   : {EDA_DIR}")
print(f"Python {sys.version.split()[0]} | numpy {np.__version__} | pandas {pd.__version__}")
print(f"matplotlib {matplotlib.__version__} | seaborn {sns.__version__}")

---
## Task A — Dataset Verification

### A1. Load and Validate `profile.txt`

The profile file has 2,205 rows (one per operating cycle) and 5 columns:
- col 0: Cooler condition (%)
- col 1: Valve condition (%)
- **col 2: Internal pump leakage ← classification target**
- col 3: Hydraulic accumulator (bar)
- col 4: Stable flag

In [ ]:
profile = load_profile(RAW_DIR)

print(f"profile.txt shape : {profile.shape}  (expected (2205, 5))")
print(f"Columns           : {list(profile.columns)}")
print()
print("First 10 rows:")
display(profile.head(10))

# Verify label values
actual_labels = sorted(profile["pump_leakage"].unique())
print(f"\nUnique pump_leakage values : {actual_labels}")
assert actual_labels == [0, 1, 2], f"Unexpected values: {actual_labels}"
print("✓ pump_leakage values confirmed: {0, 1, 2}")

### A2. Verify Target Label Mapping

In [ ]:
print("Internal pump leakage → SmartPump class mapping (confirmed from docs/description.txt):")
print()
print("  Raw value  |  Class    |  Meaning")
print("  -----------+-----------+--------------------------")
print("  0          |  Normal   |  No internal pump leakage")
print("  1          |  Warning  |  Weak internal pump leakage")
print("  2          |  Fault    |  Severe internal pump leakage")
print()

describe_dataset(profile)

### A3. Verify Class Distribution

In [ ]:
n_total    = len(profile)
n_stable   = int((profile["stable_flag"] == 0).sum())
n_unstable = int((profile["stable_flag"] == 1).sum())

print(f"Total cycles   : {n_total}")
print(f"Stable cycles  : {n_stable}  ({100*n_stable/n_total:.1f}%)")
print(f"Unstable cycles: {n_unstable}  ({100*n_unstable/n_total:.1f}%)")
print()

class_df = pd.DataFrame([
    {
        "class_code": code,
        "class_name": name,
        "count_all": int((profile["pump_leakage"] == code).sum()),
        "count_stable": int(((profile["pump_leakage"] == code) & (profile["stable_flag"] == 0)).sum()),
        "count_unstable": int(((profile["pump_leakage"] == code) & (profile["stable_flag"] == 1)).sum()),
    }
    for code, name in LABEL_MAP.items()
])
class_df["pct_all"] = (class_df["count_all"] / n_total * 100).round(1)
class_df["pct_stable"] = (class_df["count_stable"] / n_stable * 100).round(1)
class_df["pct_unstable"] = (class_df["count_unstable"] / n_unstable * 100).round(1)

print("Class distribution (all cycles):")
display(class_df)

class_df.to_csv(EDA_DIR / "class_distribution.csv", index=False)
print("\n✓ Saved: class_distribution.csv")

### A4. Validate All Sensor File Shapes

In [ ]:
print(f"{'Sensor File':<12}  {'Type':<12}  {'Rate (Hz)':<10}  {'Expected Shape':<18}  {'Actual Shape':<18}  Status")
print("-" * 92)

shape_results = []
all_ok = True

for sensor, meta in SENSOR_META.items():
    fname    = f"{sensor}.txt"
    expected = EXPECTED_SHAPES[fname]
    try:
        df = load_sensor(RAW_DIR, fname)
        ok = df.shape == expected
        status = "✓ OK" if ok else "✗ MISMATCH"
        if not ok:
            all_ok = False
        shape_results.append({
            "file": fname, "type": meta["type"], "rate_hz": meta["rate_hz"],
            "expected": str(expected), "actual": str(df.shape), "ok": ok,
        })
    except FileNotFoundError:
        status = "✗ MISSING"
        all_ok = False
        shape_results.append({
            "file": fname, "type": meta["type"], "rate_hz": meta["rate_hz"],
            "expected": str(expected), "actual": "NOT FOUND", "ok": False,
        })
    print(f"{fname:<12}  {meta['type']:<12}  {meta['rate_hz']:<10}  {str(expected):<18}  {str(shape_results[-1]['actual']):<18}  {status}")

print()
print(f"All sensor files valid: {all_ok}")

assert all_ok, "One or more sensor files failed validation!"

### A5. Dataset Summary — Save to CSV

In [ ]:
n_per_class = {k: int((profile["pump_leakage"] == k).sum()) for k in [0, 1, 2]}

summary_rows = [
    ("dataset_name",         "Condition Monitoring of Hydraulic Systems"),
    ("source",               "UCI Machine Learning Repository (ID 447)"),
    ("doi",                  "10.24432/C5CW21"),
    ("license",              "CC BY 4.0"),
    ("n_cycles_total",       n_total),
    ("n_cycles_stable",      n_stable),
    ("n_cycles_unstable",    n_unstable),
    ("pct_stable",           f"{100*n_stable/n_total:.1f}"),
    ("n_sensor_files",       17),
    ("target_file",          "profile.txt"),
    ("target_column",        "pump_leakage (0-indexed col 2)"),
    ("n_classes",            3),
    ("class_Normal_code",    0),
    ("class_Normal_count",   n_per_class[0]),
    ("class_Warning_code",   1),
    ("class_Warning_count",  n_per_class[1]),
    ("class_Fault_code",     2),
    ("class_Fault_count",    n_per_class[2]),
    ("cycle_duration_s",     60),
    ("max_rate_hz",          100),
    ("min_rate_hz",          1),
    ("missing_values",       0),
    ("profile_verified",     True),
]

summary_df = pd.DataFrame(summary_rows, columns=["property", "value"])
summary_df.to_csv(EDA_DIR / "dataset_summary.csv", index=False)
print("Dataset summary:")
display(summary_df)
print("\n✓ Saved: dataset_summary.csv")

---
## Task B — Statistical Analysis

### B1. Compute Cycle-Level Means

To efficiently analyze 17 sensors (each with up to 6,000 columns), we reduce each cycle's time-series to its **per-cycle mean**. This is used for statistical analysis and visualization only — not as the final feature representation.

In [ ]:
print("Loading all sensor files and computing cycle-level means...")
print("(Each sensor file is loaded as float32 to reduce memory usage)\n")

cycle_means        = {}
within_cycle_stds  = {}
quality_rows       = []

for sensor, meta in SENSOR_META.items():
    path = RAW_DIR / f"{sensor}.txt"
    arr  = pd.read_csv(path, sep="\t", header=None, dtype=np.float32).values

    expected_shape = (2205, meta["n_cols"])
    shape_ok       = arr.shape == expected_shape
    n_missing      = int(np.isnan(arr).sum())
    n_inf          = int(np.isinf(arr).sum())
    row_std        = arr.std(axis=1)
    n_const_cycles = int((row_std == 0).sum())  # cycles where all readings identical

    cycle_means[sensor]       = arr.mean(axis=1).astype(np.float64)
    within_cycle_stds[sensor] = float(row_std.mean())

    quality_rows.append({
        "sensor":             sensor,
        "unit":               meta["unit"],
        "type":               meta["type"],
        "rate_hz":            meta["rate_hz"],
        "expected_shape":     str(expected_shape),
        "actual_shape":       str(arr.shape),
        "shape_ok":           shape_ok,
        "global_min":         round(float(np.nanmin(arr)), 4),
        "global_max":         round(float(np.nanmax(arr)), 4),
        "n_missing":          n_missing,
        "n_inf":              n_inf,
        "n_constant_cycles":  n_const_cycles,
        "any_issues":         not shape_ok or n_missing > 0 or n_inf > 0,
    })

    status = "✓" if quality_rows[-1]["any_issues"] is False else "⚠"
    print(f"  {status} {sensor:5s}  shape={arr.shape}  missing={n_missing}  inf={n_inf}  const_cycles={n_const_cycles}")
    del arr

# Assemble unified cycle-level DataFrame
cm_df = pd.DataFrame(cycle_means, index=range(2205))
cm_df["pump_leakage"] = profile["pump_leakage"].values
cm_df["class_name"]   = cm_df["pump_leakage"].map(LABEL_MAP)
cm_df["stable_flag"]  = profile["stable_flag"].values

print(f"\nCycle-means DataFrame shape: {cm_df.shape}")
print("(Each row = one 60-second cycle; columns = per-cycle mean per sensor)")

### B2. Data Quality Report — Save to CSV

In [ ]:
quality_df = pd.DataFrame(quality_rows)
quality_df.to_csv(EDA_DIR / "data_quality_report.csv", index=False)

print("Data quality report:")
display(quality_df[[
    "sensor", "type", "rate_hz", "global_min", "global_max",
    "n_missing", "n_inf", "n_constant_cycles", "shape_ok"
]])

total_missing = quality_df["n_missing"].sum()
total_inf     = quality_df["n_inf"].sum()
total_const   = quality_df["n_constant_cycles"].sum()
print(f"\nTotal missing values     : {total_missing}")
print(f"Total infinite values    : {total_inf}")
print(f"Total constant-row cycles: {total_const}")
print("\n✓ Saved: data_quality_report.csv")

### B3. Descriptive Statistics — All Sensors

In [ ]:
def describe_arr(arr: np.ndarray) -> dict:
    """Compute descriptive statistics on a 1-D numeric array."""
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return {k: np.nan for k in ["n","mean","median","std","min","q25","q75","max","iqr","n_missing","n_inf"]}
    return {
        "n":         len(arr),
        "mean":      float(np.mean(arr)),
        "median":    float(np.median(arr)),
        "std":       float(np.std(arr, ddof=1)),
        "min":       float(np.min(arr)),
        "q25":       float(np.percentile(arr, 25)),
        "q75":       float(np.percentile(arr, 75)),
        "max":       float(np.max(arr)),
        "iqr":       float(np.percentile(arr, 75) - np.percentile(arr, 25)),
        "n_missing": int(np.isnan(arr).sum()),
        "n_inf":     int(np.isinf(arr).sum()),
    }

sensor_stats = []
for sensor, meta in SENSOR_META.items():
    s = describe_arr(cm_df[sensor].values)
    s["sensor"]  = sensor
    s["unit"]    = meta["unit"]
    s["type"]    = meta["type"]
    s["rate_hz"] = meta["rate_hz"]
    sensor_stats.append(s)

col_order = ["sensor", "unit", "type", "rate_hz",
             "n", "mean", "median", "std", "min", "q25", "q75", "max", "iqr",
             "n_missing", "n_inf"]
stats_df = pd.DataFrame(sensor_stats)[col_order]
stats_df.to_csv(EDA_DIR / "sensor_statistics.csv", index=False)

print("Sensor descriptive statistics (cycle-mean):")
display(stats_df.set_index("sensor").round(4))
print("\n✓ Saved: sensor_statistics.csv")

### B4. Per-Class Statistics — Sensor Measurements by Pump Condition

In [ ]:
sensor_class_stats = []
for sensor, meta in SENSOR_META.items():
    arr = cm_df[sensor].values
    for code, name in LABEL_MAP.items():
        mask    = cm_df["pump_leakage"] == code
        s_class = describe_arr(arr[mask])
        s_class["sensor"]     = sensor
        s_class["class_code"] = code
        s_class["class_name"] = name
        s_class["unit"]       = meta["unit"]
        sensor_class_stats.append(s_class)

cls_col_order = ["sensor", "class_code", "class_name", "unit",
                 "n", "mean", "median", "std", "min", "q25", "q75", "max", "iqr"]
cls_stats_df  = pd.DataFrame(sensor_class_stats)[cls_col_order]
cls_stats_df.to_csv(EDA_DIR / "sensor_class_statistics.csv", index=False)

# Show FS1 and SE as examples (highest and notable inter-class difference)
print("Per-class statistics for selected sensors (FS1, SE, EPS1):")
display(
    cls_stats_df[cls_stats_df["sensor"].isin(["FS1", "SE", "EPS1"])]
    .set_index(["sensor", "class_name"])
    [["n", "mean", "median", "std", "min", "max"]]
    .round(3)
)
print("\n✓ Saved: sensor_class_statistics.csv")

### B5. Correlation Matrix

In [ ]:
corr_df = cm_df[SENSOR_ORDER].corr(method="pearson")
corr_df.to_csv(EDA_DIR / "correlation_matrix.csv")

# Report highly correlated pairs
print("Sensor pairs with |Pearson r| > 0.80 (cycle-mean):")
print(f"{'Sensor A':<8}  {'Sensor B':<8}  {'r':>8}")
print("-" * 30)
high_corr = []
for i, s1 in enumerate(SENSOR_ORDER):
    for j, s2 in enumerate(SENSOR_ORDER):
        if j > i:
            r = corr_df.loc[s1, s2]
            if abs(r) > 0.80:
                high_corr.append((s1, s2, round(r, 3)))
high_corr.sort(key=lambda x: abs(x[2]), reverse=True)
for s1, s2, r in high_corr:
    print(f"{s1:<8}  {s2:<8}  {r:+.3f}")

print(f"\nTotal high-correlation pairs: {len(high_corr)}")
print("\n✓ Saved: correlation_matrix.csv")

---
## Task C — Data Visualization

### C1. Class Distribution

In [ ]:
counts_all = [n_per_class[k] for k in [0, 1, 2]]
colors     = [CLASS_COLORS[n] for n in CLASS_ORDER]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

bars = axes[0].bar(CLASS_ORDER, counts_all, color=colors, edgecolor="white", linewidth=1.5, zorder=3)
axes[0].yaxis.grid(True, alpha=0.5, zorder=0)
axes[0].set_axisbelow(True)
for bar, val in zip(bars, counts_all):
    pct = 100 * val / n_total
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 18,
                 f"{val:,}\n({pct:.1f}%)",
                 ha="center", va="bottom", fontsize=10, fontweight="bold")
axes[0].axhline(n_total / 3, color="grey", linestyle="--", linewidth=1.2, alpha=0.7,
                label=f"Balanced baseline ({int(n_total/3)})")
axes[0].set_ylim(0, 1600)
axes[0].set_title("Pump Leakage Class Distribution\n(All 2,205 Cycles)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("SmartPump Class", fontsize=11)
axes[0].set_ylabel("Number of Cycles", fontsize=11)
axes[0].legend(fontsize=9)

axes[1].pie(counts_all,
            labels=[f"{n}\n({v:,})" for n, v in zip(CLASS_ORDER, counts_all)],
            autopct="%1.1f%%", colors=colors, startangle=90,
            wedgeprops={"edgecolor": "white", "linewidth": 2},
            textprops={"fontsize": 10})
axes[1].set_title("Class Proportions", fontsize=12, fontweight="bold")

plt.suptitle("SmartPump AI — Pump Leakage Class Distribution", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(VIZ_DIR / "01_class_distribution.png")
plt.show()
print("✓ Saved: 01_class_distribution.png")

### C2. Stable Flag Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Overall stable / unstable
s_vals   = [n_stable, n_unstable]
s_labels = ["Stable\n(flag = 0)", "Not Yet Stable\n(flag = 1)"]
s_colors = ["#43A047", "#E53935"]
bars2    = axes[0].bar(s_labels, s_vals, color=s_colors, edgecolor="white", linewidth=1.5, zorder=3)
axes[0].yaxis.grid(True, alpha=0.5, zorder=0)
axes[0].set_axisbelow(True)
for bar, val in zip(bars2, s_vals):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 20,
                 f"{val:,}\n({100*val/n_total:.1f}%)",
                 ha="center", va="bottom", fontsize=11, fontweight="bold")
axes[0].set_title("Stable Flag Distribution\n(All 2,205 Cycles)", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Number of Cycles", fontsize=11)
axes[0].set_ylim(0, 1800)

# Class breakdown inside stable vs unstable
x     = np.arange(3)
width = 0.35
stab_counts   = [int(((profile["pump_leakage"] == k) & (profile["stable_flag"] == 0)).sum()) for k in [0,1,2]]
unstab_counts = [int(((profile["pump_leakage"] == k) & (profile["stable_flag"] == 1)).sum()) for k in [0,1,2]]

r1 = axes[1].bar(x - width/2, stab_counts,   width, label="Stable (flag=0)",    color="#43A047", alpha=0.85, edgecolor="white", zorder=3)
r2 = axes[1].bar(x + width/2, unstab_counts, width, label="Not Stable (flag=1)", color="#E53935", alpha=0.85, edgecolor="white", zorder=3)
axes[1].yaxis.grid(True, alpha=0.5, zorder=0)
axes[1].set_axisbelow(True)
for rect, val in list(zip(r1, stab_counts)) + list(zip(r2, unstab_counts)):
    axes[1].text(rect.get_x() + rect.get_width() / 2, rect.get_height() + 5,
                 str(val), ha="center", va="bottom", fontsize=9, fontweight="bold")
axes[1].set_xticks(x)
axes[1].set_xticklabels(CLASS_ORDER, fontsize=11)
axes[1].set_title("Class Distribution Within\nStable vs. Unstable Cycles", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Number of Cycles", fontsize=11)
axes[1].legend(fontsize=10)

plt.suptitle("SmartPump AI — Stable Flag Analysis", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(VIZ_DIR / "02_stable_flag_analysis.png")
plt.show()

print(f"Stable class distribution  : Normal={stab_counts[0]}, Warning={stab_counts[1]}, Fault={stab_counts[2]}")
print(f"Unstable class distribution: Normal={unstab_counts[0]}, Warning={unstab_counts[1]}, Fault={unstab_counts[2]}")
print("⚠ Note: 96.8% of unstable cycles are Normal — this is a critical class imbalance risk.")
print("✓ Saved: 02_stable_flag_analysis.png")

### C3. Box Plots — Pressure Sensors & Motor Power by Class

In [ ]:
pressure_sensors = ["PS1", "PS2", "PS3", "PS4", "PS5", "PS6", "EPS1"]
fig, axes = plt.subplots(1, 7, figsize=(21, 5))

for ax, sensor in zip(axes, pressure_sensors):
    for idx, (code, cname) in enumerate(LABEL_MAP.items()):
        vals = cm_df.loc[cm_df["pump_leakage"] == code, sensor].values
        bp = ax.boxplot(vals, positions=[idx], widths=0.55,
                        patch_artist=True,
                        medianprops={"color": "black", "linewidth": 2},
                        whiskerprops={"linewidth": 1.1},
                        capprops={"linewidth": 1.1},
                        flierprops={"marker": ".", "markersize": 2.5, "alpha": 0.35})
        bp["boxes"][0].set_facecolor(list(CLASS_COLORS.values())[idx])
        bp["boxes"][0].set_alpha(0.80)
    unit = SENSOR_META[sensor]["unit"]
    ax.set_title(sensor, fontsize=11, fontweight="bold")
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(["N", "W", "F"], fontsize=8)
    ax.set_xlabel("N/W/F", fontsize=7)
    ax.set_ylabel(unit if ax is axes[0] else "", fontsize=9)
    ax.grid(axis="y", alpha=0.4)

legend_handles = [plt.Rectangle((0, 0), 1, 1, color=c, alpha=0.8) for c in CLASS_COLORS.values()]
fig.legend(legend_handles, CLASS_ORDER, loc="upper right", fontsize=10, ncol=3,
           bbox_to_anchor=(1.0, 1.05))
plt.suptitle("Pressure Sensors (PS1–PS6) & Motor Power (EPS1) — Cycle-Mean by Class",
             fontsize=12, fontweight="bold", y=1.06)
plt.tight_layout()
plt.savefig(VIZ_DIR / "03_boxplots_pressure_power.png")
plt.show()
print("✓ Saved: 03_boxplots_pressure_power.png")

### C4. Box Plots — Flow & Vibration by Class

In [ ]:
flow_vib = ["FS1", "FS2", "VS1"]
fig, axes = plt.subplots(1, 3, figsize=(11, 5))

for ax, sensor in zip(axes, flow_vib):
    for idx, (code, cname) in enumerate(LABEL_MAP.items()):
        vals = cm_df.loc[cm_df["pump_leakage"] == code, sensor].values
        bp = ax.boxplot(vals, positions=[idx], widths=0.55,
                        patch_artist=True,
                        medianprops={"color": "black", "linewidth": 2.2},
                        whiskerprops={"linewidth": 1.2},
                        capprops={"linewidth": 1.2},
                        flierprops={"marker": ".", "markersize": 3, "alpha": 0.4})
        bp["boxes"][0].set_facecolor(list(CLASS_COLORS.values())[idx])
        bp["boxes"][0].set_alpha(0.80)
    unit = SENSOR_META[sensor]["unit"]
    ax.set_title(sensor, fontsize=12, fontweight="bold")
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(CLASS_ORDER, fontsize=9)
    ax.set_ylabel(unit, fontsize=10)
    ax.grid(axis="y", alpha=0.4)

legend_handles = [plt.Rectangle((0, 0), 1, 1, color=c, alpha=0.8) for c in CLASS_COLORS.values()]
fig.legend(legend_handles, CLASS_ORDER, loc="upper right", fontsize=10, ncol=3,
           bbox_to_anchor=(1.0, 1.05))
plt.suptitle("Flow Sensors (FS1, FS2) & Vibration (VS1) — Cycle-Mean by Class",
             fontsize=12, fontweight="bold", y=1.05)
plt.tight_layout()
plt.savefig(VIZ_DIR / "04_boxplots_flow_vibration.png")
plt.show()
print("✓ Saved: 04_boxplots_flow_vibration.png")

### C5. Box Plots — Temperature & Virtual Sensors by Class

In [ ]:
temp_virt = ["TS1", "TS2", "TS3", "TS4", "CE", "CP", "SE"]
fig, axes = plt.subplots(1, 7, figsize=(21, 5))

for ax, sensor in zip(axes, temp_virt):
    for idx, (code, cname) in enumerate(LABEL_MAP.items()):
        vals = cm_df.loc[cm_df["pump_leakage"] == code, sensor].values
        bp = ax.boxplot(vals, positions=[idx], widths=0.55,
                        patch_artist=True,
                        medianprops={"color": "black", "linewidth": 2},
                        whiskerprops={"linewidth": 1.1},
                        capprops={"linewidth": 1.1},
                        flierprops={"marker": ".", "markersize": 2.5, "alpha": 0.35})
        bp["boxes"][0].set_facecolor(list(CLASS_COLORS.values())[idx])
        bp["boxes"][0].set_alpha(0.80)
    unit = SENSOR_META[sensor]["unit"]
    ax.set_title(sensor, fontsize=11, fontweight="bold")
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(["N", "W", "F"], fontsize=8)
    ax.set_xlabel("N/W/F", fontsize=7)
    ax.set_ylabel(unit if ax is axes[0] else "", fontsize=9)
    ax.grid(axis="y", alpha=0.4)

legend_handles = [plt.Rectangle((0, 0), 1, 1, color=c, alpha=0.8) for c in CLASS_COLORS.values()]
fig.legend(legend_handles, CLASS_ORDER, loc="upper right", fontsize=10, ncol=3,
           bbox_to_anchor=(1.0, 1.05))
plt.suptitle("Temperature Sensors (TS1–TS4) & Virtual Sensors (CE, CP, SE) — Cycle-Mean by Class",
             fontsize=12, fontweight="bold", y=1.06)
plt.tight_layout()
plt.savefig(VIZ_DIR / "05_boxplots_temperature_virtual.png")
plt.show()
print("✓ Saved: 05_boxplots_temperature_virtual.png")

### C6. Raw Time-Series Plots — FS1 & TS1

One representative 60-second cycle per class for FS1 (flow, 10 Hz) and TS1 (temperature, 1 Hz).

In [ ]:
fs1_arr = pd.read_csv(RAW_DIR / "FS1.txt", sep="\t", header=None, dtype=np.float32).values
ts1_arr = pd.read_csv(RAW_DIR / "TS1.txt", sep="\t", header=None, dtype=np.float32).values

normal_idx  = int(profile[profile["pump_leakage"] == 0].index[0])
warning_idx = int(profile[profile["pump_leakage"] == 1].index[0])
fault_idx   = int(profile[profile["pump_leakage"] == 2].index[0])

t_fs1 = np.linspace(0, 60, 600)   # 10 Hz × 60 s
t_ts1 = np.linspace(0.5, 60, 60)  # 1 Hz × 60 s

fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharey="row")
cycle_meta = [
    (normal_idx,  "Normal",  "#2196F3"),
    (warning_idx, "Warning", "#FF9800"),
    (fault_idx,   "Fault",   "#F44336"),
]

for col, (cidx, clabel, ccolor) in enumerate(cycle_meta):
    ax = axes[0, col]
    ax.plot(t_fs1, fs1_arr[cidx], color=ccolor, linewidth=0.85, alpha=0.9)
    ax.set_title(f"FS1 — {clabel}\n(Cycle #{cidx})", fontsize=10, fontweight="bold")
    ax.set_xlabel("Time within cycle (s)", fontsize=9)
    ax.set_ylabel("Volume Flow (l/min)" if col == 0 else "", fontsize=9)
    ax.grid(alpha=0.3)

    ax = axes[1, col]
    ax.plot(t_ts1, ts1_arr[cidx], color=ccolor, linewidth=1.5,
            marker="o", markersize=4, markerfacecolor="white", markeredgewidth=1.2)
    ax.set_title(f"TS1 — {clabel}\n(Cycle #{cidx})", fontsize=10, fontweight="bold")
    ax.set_xlabel("Time within cycle (s)", fontsize=9)
    ax.set_ylabel("Temperature (°C)" if col == 0 else "", fontsize=9)
    ax.grid(alpha=0.3)

del fs1_arr, ts1_arr
plt.suptitle(
    "Raw Sensor Time-Series: Flow (FS1, 10 Hz) & Temperature (TS1, 1 Hz)\n"
    "One representative 60-second cycle per class",
    fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(VIZ_DIR / "06_timeseries_comparison.png")
plt.show()
print("✓ Saved: 06_timeseries_comparison.png")

### C7. Sensor Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(
    corr_df,
    annot=True, fmt=".2f", annot_kws={"size": 7.5},
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.4,
    cbar_kws={"shrink": 0.78, "label": "Pearson r"},
    ax=ax
)
ax.set_title("Sensor Cycle-Mean Pearson Correlation Matrix\n(All 2,205 cycles)",
             fontsize=13, fontweight="bold", pad=14)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(VIZ_DIR / "07_correlation_heatmap.png")
plt.show()
print("✓ Saved: 07_correlation_heatmap.png")

### C8. Cycle-Mean Histograms — Key Sensors by Class

In [ ]:
key_sensors = ["FS1", "SE", "EPS1", "PS1"]
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes_flat = axes.flatten()

for ax, sensor in zip(axes_flat, key_sensors):
    for code, cname in LABEL_MAP.items():
        vals = cm_df.loc[cm_df["pump_leakage"] == code, sensor].values
        ax.hist(vals, bins=35, alpha=0.60, label=cname,
                color=CLASS_COLORS[cname], edgecolor="none", density=False)
    unit = SENSOR_META[sensor]["unit"]
    ax.set_title(f"{sensor} — Cycle-Mean Distribution by Class", fontsize=11, fontweight="bold")
    ax.set_xlabel(f"Cycle mean ({unit})", fontsize=10)
    ax.set_ylabel("Count (cycles)", fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.4)

plt.suptitle("Cycle-Mean Histograms: Key Sensors — Normal vs Warning vs Fault",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(VIZ_DIR / "08_sensor_histograms.png")
plt.show()
print("✓ Saved: 08_sensor_histograms.png")

### C9. All 17 Sensors Overview — Box Plots

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(22, 16))
axes_flat = axes.flatten()

for ax_idx, sensor in enumerate(SENSOR_ORDER):
    ax = axes_flat[ax_idx]
    for idx, (code, cname) in enumerate(LABEL_MAP.items()):
        vals = cm_df.loc[cm_df["pump_leakage"] == code, sensor].values
        bp = ax.boxplot(vals, positions=[idx], widths=0.55,
                        patch_artist=True,
                        medianprops={"color": "black", "linewidth": 1.8},
                        whiskerprops={"linewidth": 1.0},
                        capprops={"linewidth": 1.0},
                        flierprops={"marker": ".", "markersize": 2, "alpha": 0.3})
        bp["boxes"][0].set_facecolor(list(CLASS_COLORS.values())[idx])
        bp["boxes"][0].set_alpha(0.80)
    unit = SENSOR_META[sensor]["unit"]
    ax.set_title(f"{sensor}\n({unit})", fontsize=9, fontweight="bold")
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(["N", "W", "F"], fontsize=7)
    ax.tick_params(axis="y", labelsize=7)
    ax.grid(axis="y", alpha=0.35)

# Hide empty subplot panels (17 sensors, 20 slots)
for ax_idx in range(len(SENSOR_ORDER), len(axes_flat)):
    axes_flat[ax_idx].set_visible(False)

legend_handles = [plt.Rectangle((0, 0), 1, 1, color=c, alpha=0.8) for c in CLASS_COLORS.values()]
fig.legend(legend_handles, CLASS_ORDER, loc="lower right", fontsize=11, ncol=3,
           bbox_to_anchor=(0.98, 0.02))
plt.suptitle("All 17 Sensors — Cycle-Mean Distribution by Pump Leakage Class\n"
             "(N = Normal  W = Warning  F = Fault)",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(VIZ_DIR / "09_all_sensors_overview.png")
plt.show()
print("✓ Saved: 09_all_sensors_overview.png")

---
## Task D — Sensor Analysis

### D1. Inter-Class Separation — Effect Size (Cohen's d)

To quantify which sensors best separate the three classes, we compute Cohen's d between Normal and Fault classes for each sensor. This is a preliminary indicator only — it does not constitute a proof of predictive importance.

In [ ]:
def cohens_d(group1: np.ndarray, group2: np.ndarray) -> float:
    """Compute pooled Cohen's d between two groups."""
    n1, n2    = len(group1), len(group2)
    s1, s2    = np.std(group1, ddof=1), np.std(group2, ddof=1)
    s_pooled  = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
    if s_pooled == 0:
        return 0.0
    return (np.mean(group1) - np.mean(group2)) / s_pooled

normal_mask  = cm_df["pump_leakage"] == 0
warning_mask = cm_df["pump_leakage"] == 1
fault_mask   = cm_df["pump_leakage"] == 2

separation = []
for sensor in SENSOR_ORDER:
    g_normal  = cm_df.loc[normal_mask, sensor].values
    g_warning = cm_df.loc[warning_mask, sensor].values
    g_fault   = cm_df.loc[fault_mask, sensor].values

    d_nw = abs(cohens_d(g_normal, g_warning))
    d_nf = abs(cohens_d(g_normal, g_fault))
    d_wf = abs(cohens_d(g_warning, g_fault))

    # Mann-Whitney U test (non-parametric, appropriate for cycle-means)
    u_nf, p_nf = scipy_stats.mannwhitneyu(g_normal, g_fault, alternative="two-sided")

    separation.append({
        "sensor":       sensor,
        "type":         SENSOR_META[sensor]["type"],
        "unit":         SENSOR_META[sensor]["unit"],
        "d_Normal_Warning": round(d_nw, 3),
        "d_Normal_Fault":   round(d_nf, 3),
        "d_Warning_Fault":  round(d_wf, 3),
        "max_d":             round(max(d_nw, d_nf, d_wf), 3),
        "p_Normal_vs_Fault": float(p_nf),
    })

sep_df = pd.DataFrame(separation).sort_values("max_d", ascending=False)

print("Sensor inter-class separation (Cohen's d, cycle-means):")
print("Interpretation: d>0.8 = large effect, 0.5<d<0.8 = medium, d<0.2 = negligible")
print()
display(sep_df.set_index("sensor")[["type", "unit", "d_Normal_Warning", "d_Normal_Fault", "d_Warning_Fault", "max_d", "p_Normal_vs_Fault"]])

### D2. Sensor Informativeness Interpretation

In [ ]:
print("Sensor analysis findings (observed from cycle-mean statistics):")
print("=" * 70)
print()

highly_sep = sep_df[sep_df["max_d"] >= 0.5][["sensor", "type", "max_d"]]
mod_sep    = sep_df[(sep_df["max_d"] >= 0.2) & (sep_df["max_d"] < 0.5)][["sensor", "type", "max_d"]]
low_sep    = sep_df[sep_df["max_d"] < 0.2][["sensor", "type", "max_d"]]

print(f"High inter-class separation (max Cohen's d >= 0.5):")
if len(highly_sep) > 0:
    for _, row in highly_sep.iterrows():
        print(f"  {row['sensor']:<6} ({row['type']:<12})  max_d = {row['max_d']:.3f}")
else:
    print("  None")

print(f"\nModerate inter-class separation (0.2 <= max_d < 0.5):")
if len(mod_sep) > 0:
    for _, row in mod_sep.iterrows():
        print(f"  {row['sensor']:<6} ({row['type']:<12})  max_d = {row['max_d']:.3f}")
else:
    print("  None")

print(f"\nLow separation (max_d < 0.2):")
if len(low_sep) > 0:
    for _, row in low_sep.iterrows():
        print(f"  {row['sensor']:<6} ({row['type']:<12})  max_d = {row['max_d']:.3f}")
else:
    print("  None")

print()
print("⚠ CAUTION: Cohen's d on cycle-means is a crude approximation.")
print("  Actual predictive importance must be established with feature extraction")
print("  and model-based feature importance (e.g., Random Forest Gini importance).")
print("  Low d does not mean the sensor is useless — higher-order statistics")
print("  (std, skewness, IQR) may reveal differences invisible in means.")

---
## Task E — Data Quality & Engineering Risks

### E1. Missing Values, Constant Cycles, Outliers

In [ ]:
print("=" * 60)
print("DATA QUALITY FINDINGS")
print("=" * 60)

total_missing = quality_df["n_missing"].sum()
total_inf     = quality_df["n_inf"].sum()
total_const   = quality_df["n_constant_cycles"].sum()

print(f"\n1. Missing values across all sensor files: {total_missing}")
if total_missing == 0:
    print("   ✓ No missing values detected. Consistent with UCI documentation.")

print(f"\n2. Infinite values across all sensor files: {total_inf}")
if total_inf == 0:
    print("   ✓ No infinite values detected.")

print(f"\n3. Constant-row cycles (zero within-cycle variance):")
const_sensors = quality_df[quality_df["n_constant_cycles"] > 0][["sensor", "n_constant_cycles"]]
if len(const_sensors) > 0:
    display(const_sensors)
    print("   ⚠ PS4 has 1,238 constant-row cycles — many cycles measure 0 bar consistently.")
    print("     This is physically expected: PS4 measures accumulator pressure and may")
    print("     be near-zero when the accumulator is not engaged.")
else:
    print("   ✓ No constant-row cycles.")

print(f"\n4. All 17 sensor files validated with correct shapes: ✓")

### E2. Class Imbalance Analysis

In [ ]:
print("CLASS IMBALANCE ANALYSIS")
print("-" * 40)
print(f"Normal : {n_per_class[0]:5d} cycles ({100*n_per_class[0]/n_total:.1f}%)")
print(f"Warning: {n_per_class[1]:5d} cycles ({100*n_per_class[1]/n_total:.1f}%)")
print(f"Fault  : {n_per_class[2]:5d} cycles ({100*n_per_class[2]/n_total:.1f}%)")

imbalance_ratio = n_per_class[0] / n_per_class[1]
print(f"\nImbalance ratio (Normal:Warning) = {imbalance_ratio:.2f}:1")
print(f"Imbalance ratio (Normal:Fault)   = {n_per_class[0]/n_per_class[2]:.2f}:1")
print()
print("Engineering risks:")
print("  - Normal class is ~2.48x larger than Warning and Fault.")
print("  - A naive majority-class classifier achieves 55.4% accuracy.")
print("  - Use macro F1-score, NOT accuracy, as the primary evaluation metric.")
print("  - Consider class-weighted loss or SMOTE in preprocessing (Step 3).")

print()
print("STABLE FLAG CLASS SKEW")
print("-" * 40)
print(f"Unstable cycles: {n_unstable} total")
print(f"  Normal  in unstable: {unstab_counts[0]} ({100*unstab_counts[0]/n_unstable:.1f}%)")
print(f"  Warning in unstable: {unstab_counts[1]} ({100*unstab_counts[1]/n_unstable:.1f}%)")
print(f"  Fault   in unstable: {unstab_counts[2]} ({100*unstab_counts[2]/n_unstable:.1f}%)")
print()
print("⚠ 96.8% of unstable cycles are Normal. If unstable cycles are included")
print("  in training without correction, the model will be biased toward Normal.")
print("  Primary training should use stable cycles only (flag=0).")

### E3. Temporal Dependency — Data Leakage Risk

In [ ]:
print("TEMPORAL DEPENDENCY & DATA LEAKAGE ANALYSIS")
print("=" * 50)
print()
print("1. Time-series structure:")
print("   Each sensor file has shape (2205, N) where N = rate_hz × 60 seconds.")
print("   Each ROW is one complete 60-second operating cycle.")
print("   Each COLUMN is a time sample within that cycle.")
print()
print("2. Within-cycle non-independence:")
print("   The 60–6000 time samples in each row come from the same cycle.")
print("   They are strongly autocorrelated and NOT independent observations.")
print("   ✗ NEVER split individual time samples into train/test.")
print("   ✓ The unit of analysis is ONE ROW (one cycle).")
print()
print("3. Leakage-safe train/test split:")
print("   Split at the cycle level (rows), not at the time sample level (columns).")
print("   Use StratifiedShuffleSplit or a block-based approach.")
print("   Keep all time samples from one cycle entirely in train OR test.")
print()
print("4. Feature extraction safety:")
print("   Statistical features (mean, std, etc.) are extracted from each cycle row.")
print("   Scaler parameters (mean, std for normalization) must be fit ONLY on training cycles.")
print("   Apply the fitted scaler to test cycles without re-fitting.")
print()
print("5. Sensor cross-contamination:")
print("   PS1↔PS2 Pearson r ≈ 0.995 (near-perfect correlation).")
print("   PS2↔SE Pearson r ≈ -0.958. These highly correlated sensors")
print("   carry redundant information. This should be addressed in feature selection (Step 4).")

### E4. Sensor Noise & Sampling Rate Differences

In [ ]:
print("SAMPLING RATE HETEROGENEITY")
print("-" * 40)
rate_groups = {}
for sensor, meta in SENSOR_META.items():
    r = meta["rate_hz"]
    rate_groups.setdefault(r, []).append(sensor)

for rate, sensors_in_group in sorted(rate_groups.items()):
    n_cols = rate * 60
    print(f"  {rate:3d} Hz ({n_cols:5d} columns/cycle): {', '.join(sensors_in_group)}")

print()
print("Implications:")
print("  - Raw features cannot be compared across sensors directly.")
print("  - Statistical aggregation (mean, std, etc.) normalizes the temporal dimension.")
print("  - High-frequency sensors (PS1-PS6, EPS1 at 100 Hz) provide more within-cycle")
print("    variation information — higher-order stats (skewness, kurtosis) are meaningful.")
print("  - Low-frequency sensors (TS, VS1, CE, CP, SE at 1 Hz) have only 60 samples;")
print("    median/IQR may add limited value given the small sample size per cycle.")

print()
print("WITHIN-CYCLE SIGNAL VARIABILITY (mean cycle std across all cycles):")
print("-" * 50)
for sensor, meta in SENSOR_META.items():
    mean_within_std = within_cycle_stds[sensor]
    print(f"  {sensor:<6}: mean within-cycle std = {mean_within_std:.4f} {meta['unit']}")

---
## Task F — Dataset Preparation Findings

### F1. Correct Data Structure Summary

In [ ]:
print("DATASET PREPARATION FINDINGS FOR STEP 3")
print("=" * 60)
print()
print("1. FILE ↔ CYCLE CORRESPONDENCE")
print("   - Each sensor file: 2,205 rows × N columns (tab-delimited).")
print("   - Row i in any sensor file = cycle i in profile.txt.")
print("   - Row i in profile.txt = condition state for cycle i.")
print("   - Alignment is verified: all files have exactly 2,205 rows.")
print()
print("2. TARGET MAPPING")
print("   profile.txt column 2 (0-indexed) = pump_leakage.")
print("   Mapping: 0 = Normal, 1 = Warning, 2 = Fault.")
print("   Verified: unique values = {0, 1, 2}. No other values present.")
print()
print("3. SENSOR CHANNEL SELECTION")
print("   All 17 sensors should be included in initial feature extraction.")
print("   PS4 has 1,238 near-zero cycles — do not drop; zero is a valid measurement.")
print("   PS1↔PS2 (r=0.995) and PS2↔SE (r=-0.958): consider removing one of each pair")
print("   during feature selection (after establishing feature importances).")
print()
print("4. CYCLE SUBSET RECOMMENDATION")
print("   Primary training: stable cycles only (stable_flag=0 → 1,449 cycles).")
print("   Sensitivity analysis: include unstable cycles and compare results.")
print("   Reason: 96.8% of unstable cycles are Normal — including them biases models.")
print()
print("5. FEATURE EXTRACTION APPROACH")
print("   Unit of analysis: ONE ROW = one 60-second cycle.")
print("   Extract statistical features from each cycle's time-series.")
print("   Minimum features: mean, std, min, max, median per sensor per cycle.")
print("   Extended: add rms, skewness, kurtosis, IQR for high-frequency sensors.")
print("   This reduces (2205 × 43680) raw data to (2205 × 85–153) feature matrix.")
print()
print("6. REQUIRED PREPROCESSING STEPS (Step 3)")
print("   a. Load profile.txt and all 17 sensor files.")
print("   b. Optionally filter to stable cycles (flag=0).")
print("   c. Extract per-cycle statistical features from each sensor.")
print("   d. Assemble feature matrix (n_cycles × n_features).")
print("   e. Perform cycle-level stratified train/test split (never sample-level).")
print("   f. Fit StandardScaler on training set ONLY; transform test set.")
print("   g. Save processed features to data/processed/.")
print()
print("7. WHAT NOT TO DO")
print("   ✗ Do NOT split at the sensor time-sample level (data leakage).")
print("   ✗ Do NOT fit scalers on the full dataset before splitting.")
print("   ✗ Do NOT use raw 6000-column sensor vectors as features directly.")
print("   ✗ Do NOT evaluate using overall accuracy alone — use macro F1.")

### F2. Final Output Verification

In [ ]:
print("EDA OUTPUT VERIFICATION")
print("=" * 50)
print()

print("CSV files in results/eda/:")
for f in sorted(EDA_DIR.glob("*.csv")):
    n_rows = len(pd.read_csv(f))
    size_kb = f.stat().st_size // 1024
    print(f"  ✓  {f.name:<40}  {n_rows:>4} rows  {size_kb:>4} KB")

print()
print("Visualizations in results/eda/visualizations/:")
for f in sorted(VIZ_DIR.glob("*.png")):
    size_kb = f.stat().st_size // 1024
    print(f"  ✓  {f.name:<48}  {size_kb:>5} KB")

print()
print("Dataset integrity:")
# Verify raw files are unmodified by checking their row counts
for fname, expected_shape in EXPECTED_SHAPES.items():
    path = RAW_DIR / fname
    # Quick line count check (don't reload full data)
    with open(path, "rb") as fh:
        n_lines = sum(1 for _ in fh)
    ok = n_lines == expected_shape[0]
    status = "✓" if ok else "✗"
    print(f"  {status} {fname:<12}: {n_lines} rows (expected {expected_shape[0]})")

print()
print("✅ EDA Step 2 complete. All outputs verified. Ready for Step 3.")

---
## Summary of EDA Findings

| Finding | Detail |
|---------|--------|
| **Dataset** | 2,205 cycles × 17 sensors; UCI Condition Monitoring of Hydraulic Systems |
| **Target** | `profile.txt` col 2 — Internal pump leakage (0=Normal, 1=Warning, 2=Fault) |
| **Class distribution** | Normal 1221 (55.4%), Warning 492 (22.3%), Fault 492 (22.3%) |
| **Stable cycles** | 1,449 (65.7%); 756 unstable (96.8% Normal — severe skew) |
| **Missing values** | 0 across all 17 sensor files |
| **Constant cycles** | PS4: 1,238 cycles with zero within-cycle std (physically expected) |
| **High correlations** | PS1↔PS2 (r=0.995), PS2↔SE (r=-0.958), PS3↔FS1 (r=0.921) |
| **Best discriminators** | FS1, SE, EPS1 show largest inter-class separation in cycle-means |
| **Sampling rates** | 100 Hz (7 sensors), 10 Hz (2), 1 Hz (8) — heterogeneous |
| **Data leakage risk** | Must split at cycle level; never at time-sample level |
| **Recommended subset** | Use stable cycles only for primary training |

**Next step:** `02_preprocessing.ipynb` — Feature extraction and dataset preparation.